In [0]:
# Load events DataFrame from Unity Catalog table
from pyspark.sql import SparkSession
spark = SparkSession.getActiveSession()
events = spark.read.table("workspace.ecommerce.silver_events")



In [0]:
# Cell 1: Descriptive statistics for price
display(events.describe(["price"]))



In [0]:
# Cell 2: Hypothesis testing - weekday vs weekend conversion counts
from pyspark.sql import functions as F

events_with_weekend = events.withColumn(
    "is_weekend",
    F.dayofweek("event_time").isin([1, 7])
)
display(events_with_weekend.groupBy("is_weekend", "event_type").count())



In [0]:
# Cell 3: Correlation between price and conversion (conversion proxy: event_type == 'purchase')
from pyspark.sql import functions as F

events_with_conversion = events.withColumn(
    "conversion",
    (F.col("event_type") == "purchase").cast("double")
)
correlation = events_with_conversion.stat.corr("price", "conversion")



In [0]:
# Cell 4: Feature engineering for ML
from pyspark.sql.window import Window
from pyspark.sql import functions as F

window = Window.partitionBy("user_id").orderBy("event_time")
features = events.withColumn("hour", F.hour("event_time")) \
    .withColumn("day_of_week", F.dayofweek("event_time")) \
    .withColumn("price_log", F.log(F.col("price") + 1)) \
    .withColumn("time_since_first_view",
        F.unix_timestamp("event_time") -
        F.unix_timestamp(F.first("event_time").over(window))
    )
display(features)
